# Pendulo Simples (didatico)

Objetivo deste notebook:
1. Resolver a EDO do pendulo com Euler
2. Mostrar saida numerica inicial
3. Mostrar animacao do pendulo + grafico theta(t)
4. Repetir com Runge-Kutta 4 (RK4)

A ideia e separar o codigo em blocos claros, com comentarios explicando a logica.


Equação diferencial do pêndulo simples:

`theta'' + (g/L)*sin(theta) = 0`

Onde `theta` é o ângulo, `g` é a gravidade e `L` é o comprimento do fio.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# No Colab, usamos HTML para mostrar a animacao dentro da celula
try:
    from IPython.display import HTML
except Exception:
    HTML = None

# ============================================================
# BLOCO 1: PARAMETROS DO PROBLEMA
# ============================================================
# Parametros fisicos do sistema
g = 9.81                      # gravidade (m/s^2)
L = 1.0                       # comprimento do fio (m)

# Condicoes iniciais
theta0 = np.deg2rad(20.0)     # angulo inicial em radianos
omega0 = 0.0                  # velocidade angular inicial (rad/s)

# Parametros numericos da simulacao
dt = 0.02                     # passo de tempo (s)
t_final = 10.0                # tempo total de simulacao (s)
t = np.arange(0.0, t_final + dt, dt)
N = len(t)                    # numero total de pontos de tempo

# ============================================================
# BLOCO 2: METODO DE EULER
# ============================================================
# Logica: a cada passo, calculamos aceleracao angular
# e atualizamos velocidade e angulo com aproximacao linear.
def simular_euler():
    theta = np.zeros(N)
    omega = np.zeros(N)

    theta[0] = theta0
    omega[0] = omega0

    for i in range(N - 1):
        alpha = -(g / L) * np.sin(theta[i])
        omega[i + 1] = omega[i] + alpha * dt
        theta[i + 1] = theta[i] + omega[i] * dt

    return theta, omega

# ============================================================
# BLOCO 3: METODO RUNGE-KUTTA DE 4a ORDEM (RK4)
# ============================================================
# Logica: RK4 usa 4 estimativas por passo e combina em media
# ponderada. Em geral, e mais preciso que Euler.
def derivadas(theta, omega):
    dtheta_dt = omega
    domega_dt = -(g / L) * np.sin(theta)
    return dtheta_dt, domega_dt

def simular_rk4():
    theta = np.zeros(N)
    omega = np.zeros(N)

    theta[0] = theta0
    omega[0] = omega0

    for i in range(N - 1):
        th = theta[i]
        om = omega[i]

        k1_th, k1_om = derivadas(th, om)
        k2_th, k2_om = derivadas(th + 0.5 * dt * k1_th, om + 0.5 * dt * k1_om)
        k3_th, k3_om = derivadas(th + 0.5 * dt * k2_th, om + 0.5 * dt * k2_om)
        k4_th, k4_om = derivadas(th + dt * k3_th, om + dt * k3_om)

        theta[i + 1] = th + (dt / 6.0) * (k1_th + 2*k2_th + 2*k3_th + k4_th)
        omega[i + 1] = om + (dt / 6.0) * (k1_om + 2*k2_om + 2*k3_om + k4_om)

    return theta, omega

# ============================================================
# BLOCO 4: FUNCOES DE SAIDA (TEXTO + ANIMACAO)
# ============================================================
# Mostra tabela curta para conferir os primeiros pontos numericos
def mostrar_saida_numerica(nome_metodo, theta, omega):
    x = L * np.sin(theta)
    print(f"\nSaida numerica - {nome_metodo}")
    print(" i   t(s)    theta(rad)    omega(rad/s)    x(m)")
    for i in range(10):
        print(f"{i:2d}  {t[i]:6.3f}   {theta[i]:10.6f}    {omega[i]:11.6f}   {x[i]:8.5f}")

# Cria animacao do pendulo e, abaixo, grafico da evolucao de theta(t)
def animar_pendulo(nome_metodo, theta):
    x = L * np.sin(theta)
    y = -L * np.cos(theta)

    fig, (ax_pendulo, ax_theta) = plt.subplots(
        2, 1, figsize=(8, 7), gridspec_kw={'height_ratios': [1.3, 1.0]}
    )

    # Painel superior: pendulo oscilando
    ax_pendulo.set_title(f'{nome_metodo} - Pendulo oscilando')
    ax_pendulo.set_xlabel('x (m)')
    ax_pendulo.set_ylabel('y (m)')
    ax_pendulo.set_xlim(-1.2 * L, 1.2 * L)
    ax_pendulo.set_ylim(-1.2 * L, 0.2 * L)
    ax_pendulo.axis('equal')
    ax_pendulo.grid(alpha=0.3)
    ax_pendulo.scatter([0.0], [0.0], color='black', s=30, label='pivo')

    haste, = ax_pendulo.plot([], [], color='tab:blue', lw=2)
    massa, = ax_pendulo.plot([], [], 'o', color='tab:red', ms=12)
    trilha, = ax_pendulo.plot([], [], color='tab:orange', alpha=0.5, lw=1.5)
    ax_pendulo.legend()

    # Painel inferior: theta(t) sendo construido no tempo
    ax_theta.set_title(f'{nome_metodo} - Angulo theta(t)')
    ax_theta.set_xlabel('tempo (s)')
    ax_theta.set_ylabel('theta (rad)')
    ax_theta.set_xlim(0.0, t_final)
    margem = 0.1 * max(abs(np.min(theta)), abs(np.max(theta)), 1e-6)
    ax_theta.set_ylim(np.min(theta) - margem, np.max(theta) + margem)
    ax_theta.grid(alpha=0.3)
    linha_theta, = ax_theta.plot([], [], color='tab:green', lw=2)

    def init():
        haste.set_data([], [])
        massa.set_data([], [])
        trilha.set_data([], [])
        linha_theta.set_data([], [])
        return haste, massa, trilha, linha_theta

    def update(i):
        haste.set_data([0.0, x[i]], [0.0, y[i]])
        massa.set_data([x[i]], [y[i]])
        trilha.set_data(x[:i+1], y[:i+1])
        linha_theta.set_data(t[:i+1], theta[:i+1])
        return haste, massa, trilha, linha_theta

    # Deixa a animacao leve no Colab: limita a quantidade de frames
    max_frames = 140
    passo_frame = max(1, N // max_frames)
    frames_anim = np.arange(0, N, passo_frame)
    intervalo_ms = max(15, int(1000 * dt * passo_frame))

    ani = FuncAnimation(fig, update, frames=frames_anim, init_func=init, interval=intervalo_ms, blit=False)

    # No Colab, retornar HTML mostra a animacao na celula
    if HTML is not None:
        plt.close(fig)
        return HTML(ani.to_jshtml(default_mode='loop'))

    plt.show()
    return ani


In [ ]:
# BLOCO 5: EXECUCAO COM EULER
# Primeiro rodamos a simulacao, depois mostramos tabela e animacao.
theta_euler, omega_euler = simular_euler()
mostrar_saida_numerica('Euler', theta_euler, omega_euler)
animar_pendulo('Euler', theta_euler)


In [ ]:
# BLOCO 6: EXECUCAO COM RK4
# Mesma logica da celula anterior, agora com metodo mais preciso.
theta_rk4, omega_rk4 = simular_rk4()
mostrar_saida_numerica('Runge-Kutta 4', theta_rk4, omega_rk4)
animar_pendulo('Runge-Kutta 4', theta_rk4)
